# Start here

This notebook just proves the workspace works end to end, using the kind of data your capstone runs on: it pulls a live 7-day forecast for three parks, lands it as Parquet, loads it into DuckDB, queries it with SQL, and plots it.

Click **Run All** at the top. If VS Code asks for a kernel, pick the one with **`.venv`** in the name. Delete this notebook whenever you like.

In [ ]:
import sys

import duckdb
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
import requests

from capstone.config import RAW_DIR, WAREHOUSE, get_secret

print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | duckdb {duckdb.__version__}")

## 1. Extract: a live forecast from Open-Meteo (no API key needed)

In [ ]:
PARKS = {
    "Yosemite Valley": (37.7456, -119.5936),
    "Zion Canyon": (37.2002, -112.9870),
    "Shenandoah (Big Meadows)": (38.5293, -78.4383),
}
DAILY_FIELDS = [
    "temperature_2m_max",
    "temperature_2m_min",
    "precipitation_sum",
    "precipitation_probability_max",
    "wind_speed_10m_max",
]

frames = []
for park, (lat, lon) in PARKS.items():
    resp = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat,
            "longitude": lon,
            "daily": ",".join(DAILY_FIELDS),
            "temperature_unit": "fahrenheit",
            "wind_speed_unit": "mph",
            "precipitation_unit": "inch",
            "timezone": "auto",
            "forecast_days": 7,
        },
        timeout=30,
    )
    resp.raise_for_status()
    frames.append(pd.DataFrame(resp.json()["daily"]).assign(park=park))

forecast = pd.concat(frames, ignore_index=True).rename(columns={"time": "date"})
forecast["date"] = pd.to_datetime(forecast["date"])
forecast.to_parquet(RAW_DIR / "forecast_sample.parquet", index=False)
forecast.head()

## 2. Load + transform: the same data in DuckDB, queried with SQL

The `verdict` rule below is a placeholder, not a model — it's here to show where your real logic would plug in.

In [ ]:
con = duckdb.connect(str(WAREHOUSE))
con.register("forecast_df", forecast)
con.execute("CREATE OR REPLACE TABLE forecast AS SELECT * FROM forecast_df")

verdicts = con.sql(
    """
    SELECT
        park,
        CAST(date AS DATE)                    AS day,
        round(temperature_2m_max)             AS high_f,
        precipitation_probability_max         AS rain_chance_pct,
        round(wind_speed_10m_max)             AS wind_mph,
        CASE
            WHEN precipitation_probability_max >= 60 OR wind_speed_10m_max >= 30 THEN 'skip'
            WHEN temperature_2m_max BETWEEN 50 AND 80
                 AND precipitation_probability_max < 30 THEN 'go'
            ELSE 'maybe'
        END                                   AS verdict
    FROM forecast
    ORDER BY park, day
    """
).df()
con.close()
verdicts

## 3. Plot

In [ ]:
# Fixed series order and colors, one per park
COLORS = ["#2a78d6", "#eb6834", "#1baf7a"]
INK, MUTED, SURFACE = "#2b2b29", "#6f6e69", "#fcfcfb"

fig, ax = plt.subplots(figsize=(9, 4.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
ax.axhspan(50, 80, color="#f0efeb", zorder=0)
ax.text(
    0.005,
    79,
    "50–80°F window",
    transform=ax.get_yaxis_transform(),
    va="top",
    fontsize=9,
    color=MUTED,
)

for color, (park, grp) in zip(COLORS, forecast.groupby("park", sort=False)):
    ax.plot(
        grp["date"],
        grp["temperature_2m_max"],
        color=color,
        linewidth=2,
        marker="o",
        markersize=5,
        label=park,
        zorder=3,
    )
    last = grp.iloc[-1]
    ax.annotate(
        f"{park.split(' (')[0]}  {last.temperature_2m_max:.0f}°",
        (last["date"], last["temperature_2m_max"]),
        xytext=(6, 0),
        textcoords="offset points",
        va="center",
        fontsize=9,
        color=INK,
    )

ax.set_title("Daily high, next 7 days (°F)", loc="left", color=INK, fontsize=12)
ax.grid(axis="y", color="#e2e1dc", linewidth=0.8)
ax.set_axisbelow(True)
ax.tick_params(colors=MUTED, labelsize=9)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color("#d4d3cd")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %b %-d"))
ax.legend(
    frameon=False, fontsize=9, labelcolor=INK, loc="upper left", bbox_to_anchor=(0, -0.16), ncol=3
)
ax.margins(x=0.12)
highs = forecast["temperature_2m_max"]
ax.set_ylim(min(highs.min() - 5, 45), max(highs.max() + 5, 85))
fig.tight_layout()
plt.show()

## 4. Your API keys (optional)

If you added `NPS_API_KEY` when you created the codespace, this pulls current park alerts. If not, it tells you how to add one.

In [ ]:
nps_key = get_secret("NPS_API_KEY", required=False)
if nps_key:
    resp = requests.get(
        "https://developer.nps.gov/api/v1/alerts",
        params={"parkCode": "yose,zion,shen", "limit": 20},
        headers={"X-Api-Key": nps_key},
        timeout=30,
    )
    resp.raise_for_status()
    alerts = pd.DataFrame(resp.json()["data"])
    display(alerts[["parkCode", "category", "title"]] if len(alerts) else "No current alerts.")
else:
    print(
        'No NPS_API_KEY yet — see the README section "API keys". Everything else works without it.'
    )

## That's it

- R is set up too: open **`01_r_check.ipynb`** — it reads the Parquet file this notebook wrote.
- Code you reuse goes in `src/capstone/` and imports into any notebook.
- Raw pulls go in `data/raw/`, cleaned data in `data/processed/` (neither is committed to git).